In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
import torchvision.models as models
model=models.resnet34(weights="IMAGENET1K_V1")

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 175MB/s]


In [2]:
import numpy as np
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
train_transform=transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486,0.456,0.406],std=[0.229,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486,0.456,0.406],std=[0.229,0.224,0.225])
])
train_data=datasets.CIFAR10(root='./data',download=True,train=True,transform=train_transform)
test_data=datasets.CIFAR10(root='./data',download=True,train=False,transform=test_transform)

train_loader=DataLoader(train_data,batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
test_loader=DataLoader(test_data,batch_size=64,shuffle=False,pin_memory=True,num_workers=2)

In [5]:
for param in model.parameters():
  model.requires_grad=False

In [6]:
num_ftrs=model.fc.in_features
num_classes=10
model.fc=nn.Linear(num_ftrs,num_classes)

In [7]:
optimizer=optim.Adam(model.fc.parameters(),lr=1e-3)

In [8]:
criterion=nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=1)

In [9]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)

In [10]:
epochs=3
for epoch in range(epochs):
  model.train()
  train_loss=0
  correct = 0
  total = 0
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)

    optimizer.zero_grad()
    outputs=model(images)
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()

    train_loss+=loss.item()
    _,predicted=torch.max(outputs,1)

    correct+=(predicted==labels).sum().item()
    total+=labels.size(0)
  scheduler.step()
  train_acc = 100 * correct / total
  print(f"Epoch [{epoch+1}/{epochs}]")
  print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")

Epoch [1/3]
Train Loss: 1.7282, Train Acc: 45.13%
Epoch [2/3]
Train Loss: 1.6285, Train Acc: 49.54%
Epoch [3/3]
Train Loss: 1.6132, Train Acc: 50.50%


In [11]:
model.eval()
total=0
correct=0
test_loss=0
with torch.inference_mode():
  for images,labels in test_loader:
    images,labels=images.to(device),labels.to(device)
    output=model(images)

    loss=criterion(output,labels)
    test_loss+=loss.item()
    _,predicted=torch.max(output,1)
    correct+=(predicted==labels).sum().item()
    total+=labels.size(0)
  test_acc=100*correct/total
print(f"Test Loss: {test_loss/len(test_loader):.4f}, Test Acc: {test_acc:.2f}%")

Test Loss: 1.1884, Test Acc: 70.91%


In [12]:
import numpy as np
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [13]:
train_transform=transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486,0.456,0.406],std=[0.229,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486,0.456,0.406],std=[0.229,0.224,0.225])
])
train_data=datasets.CIFAR10(root='./data',download=True,train=True,transform=train_transform)
test_data=datasets.CIFAR10(root='./data',download=True,train=False,transform=test_transform)

train_loader1=DataLoader(train_data,batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
test_loader2=DataLoader(test_data,batch_size=64,shuffle=False,pin_memory=False,num_workers=2)

In [14]:
for param in model.layer4.parameters():
  param.requires_grad=True

In [15]:
optimizer=optim.Adam([
    {"params":model.layer4.parameters(),"lr":1e-4},
    {"params":model.fc.parameters(),"lr":1e-3}
])
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)

In [16]:
criterion=nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=1)

In [17]:
epochs = 3

for epoch in range(epochs):
    # ===== TRAIN =====
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    test_acc = 100 * correct / total

    scheduler.step()

    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {test_loss/len(test_loader):.4f}, Val   Acc: {test_acc:.2f}%")


Epoch [1/3]
Train Loss: 1.3920, Train Acc: 60.90%
Val   Loss: 2.0606, Val   Acc: 86.37%

Epoch [2/3]
Train Loss: 1.2831, Train Acc: 65.72%
Val   Loss: 2.9282, Val   Acc: 86.69%

Epoch [3/3]
Train Loss: 1.2547, Train Acc: 67.00%
Val   Loss: 3.7603, Val   Acc: 88.08%
